# 30. Dimensionality reduction and feature selection
Feature selection retains a subset of original inputs. Dimensionality reduction constructs new combinations. PCA finds orthogonal directions of large variance after centering; high variance does not guarantee predictive usefulness. TruncatedSVD can work efficiently with sparse text matrices without centering them.

Fit selection or reduction inside cross-validation when evaluating a predictor. Scale before PCA when units should not determine variance. Explained variance describes retained spread, not accuracy. Compare a reduced pipeline against a suitable unreduced baseline and inspect whether smaller size or faster prediction justifies the tradeoff.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
X = load_digits().data # Each image is represented by 64 pixel features.
prep = make_pipeline(StandardScaler(), PCA(n_components=10)) # Scale and reduce dimensionality.
Z = prep.fit_transform(X) # Build an exploratory ten-dimensional representation.
print("Original / reduced shapes:", X.shape, Z.shape) # Inspect compression.
print("Variance retained:", round(prep.named_steps["pca"].explained_variance_ratio_.sum(), 3)) # Sum retained variance ratios.
assert Z.shape[1] == 10 # Check component count.

## Four selectors inside the evaluation boundary
SelectKBest scores individual features; correlated or interacting features can challenge univariate ranking. SelectFromModel retains features using a fitted estimator’s importance or coefficients. RFE recursively removes low-ranked features and refits. SequentialFeatureSelector adds or removes features based on internal cross-validation. These wrappers can be expensive and their selections can vary with the sample. Any selection using y must remain inside the outer evaluation pipeline.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel, RFE, SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
X, y = make_classification(n_samples=120, n_features=6, n_informative=3, random_state=42) # Small enough for wrapper demonstrations.
base = LogisticRegression(max_iter=1000)
selectors = [SelectKBest(f_classif, k=3), SelectFromModel(base, threshold="median"), RFE(base, n_features_to_select=3), SequentialFeatureSelector(base, n_features_to_select=3, cv=2)]
cv = StratifiedKFold(3, shuffle=True, random_state=42)
for selector in selectors:
    pipe = make_pipeline(StandardScaler(), selector, LogisticRegression(max_iter=1000)) # Fit scaling and selection on training folds only.
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
    print(type(selector).__name__, round(scores.mean(), 3))
assert len(scores) == 3 # Three held-out evaluations per complete workflow.

### Reading the result
The scores compare workflows, not permanently discovered true feature sets. The sequential selector has an inner comparison within each outer training fold. If you use these same scores to choose the selector family, report the selected result on an untouched test or through a further nested evaluation.

### Transfer exercise
Record feature-selection frequency across folds and compare with an all-feature baseline. Report runtime and instability alongside score.

## Your turn
Compare a classifier pipeline with and without PCA on the same folds. Record score and transformed feature count.

Try the task before opening the solution on the course website.